# Library notebook — Spatial filters

**Type:** library notebook (import via `%run`; not an experiment entry point).

## Purpose

Spatial filters (Average, Median, Gaussian, Sobel, Laplacian) using manual convolution.

**Consumers:** preprocessing execution notebooks via `%run`.


# Spatial domain filters (implementation)

Each filter function is independent and receives only a grayscale `uint8` image.

Pedagogical implementation: manual convolution e explicit windows (without OpenCV).

In [ ]:
%matplotlib inline

# ==========================================================
# IMPORTS
# ==========================================================

import numpy as np

## Manual 2D convolution

Shared base for filters linearly separable (mean, Gaussian, Sobel, Laplacian).

- input uses **zero padding**;
- intermediate output em `float` antes de voltar a `uint8`.

In [ ]:
# ==========================================================
# MANUAL CONVOLUTION (GRAYSCALE)
# ==========================================================

def apply_manual_convolution(input_image, kernel):
    """
    Convolve 2D uint8 image with 2D float kernel.
    """

    # Validate input (notebook contract)
    if input_image.ndim != 2:
        raise ValueError("Image must be 2D (grayscale).")
    if input_image.dtype != np.uint8:
        raise ValueError("Image must be uint8.")

    kernel = np.asarray(kernel, dtype=np.float64)
    kernel_height, kernel_width = kernel.shape
    pad_y = kernel_height // 2
    pad_x = kernel_width // 2

    # Zero padding
    image_float = input_image.astype(np.float64)
    padded_image = np.pad(image_float, ((pad_y, pad_y), (pad_x, pad_x)), mode="constant")

    image_height, image_width = input_image.shape
    convolution_result = np.zeros((image_height, image_width), dtype=np.float64)

    for row_index in range(image_height):
        for column_index in range(image_width):
            weighted_sum = 0.0
            for kernel_row in range(kernel_height):
                for kernel_col in range(kernel_width):
                    pixel_value = padded_image[row_index + kernel_row, column_index + kernel_col]
                    kernel_weight = kernel[kernel_row, kernel_col]
                    weighted_sum += pixel_value * kernel_weight
            convolution_result[row_index, column_index] = weighted_sum

    return convolution_result


def clip_float_to_uint8(float_map):
    """Clip float values to uint8 range [0, 255]."""
    rounded_map = np.round(float_map)
    clipped_map = np.clip(rounded_map, 0, 255)
    return clipped_map.astype(np.uint8)

### Low-pass — Average (mean)

Kernel de mean $1/(N \times N)$ com odd size (ex.: 3×3).

In [ ]:
# ==========================================================
# FILTRO AVERAGE (PASSA-BAIXO)
# ==========================================================

def apply_average_filter(input_image):
    """
    Mean filter — single parameter: uint8 grayscale image.
    """

    if input_image.ndim != 2 or input_image.dtype != np.uint8:
        raise ValueError("Invalid input: expected 2D uint8 grayscale image.")

    # Janela 3x3 (fixed size per notebook contract)
    window_size = 3

    # Construir kernel de mean explicitamente
    kernel = np.ones((window_size, window_size), dtype=np.float64)
    weight_sum = window_size * window_size
    kernel = kernel / weight_sum

    filtered_map = apply_manual_convolution(input_image, kernel)
    return clip_float_to_uint8(filtered_map)

### Low-pass — Median (median)

Para cada janela, ordena valores e escolhe o elemento central.

In [ ]:
# ==========================================================
# FILTRO MEDIAN (PASSA-BAIXO)
# ==========================================================

def apply_median_filter(input_image):
    """
    Median filter — single parameter: uint8 grayscale image.
    """

    if input_image.ndim != 2 or input_image.dtype != np.uint8:
        raise ValueError("Invalid input: expected 2D uint8 grayscale image.")

    window_size = 3
    pad = window_size // 2
    image_height, image_width = input_image.shape
    padded_image = np.pad(input_image, pad, mode="edge")
    output_image_array = np.zeros_like(input_image)

    for row_index in range(image_height):
        for column_index in range(image_width):
            window_pixels = padded_image[
                row_index : row_index + window_size,
                column_index : column_index + window_size,
            ]
            sorted_values = window_pixels.reshape(-1).tolist()
            sorted_values.sort()
            median_index = len(sorted_values) // 2
            output_image_array[row_index, column_index] = sorted_values[median_index]

    return output_image_array.astype(np.uint8)

### Low-pass — Gaussian

Kernel Gaussian 2D built explicitly (sem built-in smoothing functions).

In [ ]:
# ==========================================================
# FILTRO GAUSSIAN (PASSA-BAIXO)
# ==========================================================

def apply_gaussian_filter(input_image):
    """
    Filtro Gaussian — single parameter: uint8 grayscale image.
    """

    if input_image.ndim != 2 or input_image.dtype != np.uint8:
        raise ValueError("Invalid input: expected 2D uint8 grayscale image.")

    window_size = 3
    sigma = 1.0
    centro = window_size // 2
    kernel = np.zeros((window_size, window_size), dtype=np.float64)
    kernel_sum = 0.0

    for i in range(window_size):
        for j in range(window_size):
            x = float(j - centro)
            y = float(i - centro)
            kernel_weight = np.exp(-(x * x + y * y) / (2.0 * sigma * sigma))
            kernel[i, j] = kernel_weight
            kernel_sum += kernel_weight

    kernel = kernel / kernel_sum

    filtered_map = apply_manual_convolution(input_image, kernel)
    return clip_float_to_uint8(filtered_map)

### High-pass — Sobel

Gradient in directions $G_x$ e $G_y$; magnitude $|G| = \sqrt{G_x^2 + G_y^2}$.

In [ ]:
# ==========================================================
# FILTRO SOBEL (PASSA-ALTO)
# ==========================================================

def apply_sobel_filter(input_image:
    """
    Sobel operator — single parameter: uint8 grayscale image.
    """

    if input_image.ndim != 2 or input_image.dtype != np.uint8:
        raise ValueError("Invalid input: expected 2D uint8 grayscale image.")

    kernel_gx = np.array(
        [
            [-1.0, 0.0, 1.0],
            [-2.0, 0.0, 2.0],
            [-1.0, 0.0, 1.0],
        ],
        dtype=np.float64,
    )

    kernel_gy = np.array(
        [
            [-1.0, -2.0, -1.0],
            [0.0, 0.0, 0.0],
            [1.0, 2.0, 1.0],
        ],
        dtype=np.float64,
    )

    gradient_x = apply_manual_convolution(input_image, kernel_gx)
    gradient_y = apply_manual_convolution(input_image, kernel_gy)

    image_height, image_width = input_image.shape
    magnitude = np.zeros((image_height, image_width), dtype=np.float64)

    for row_index in range(image_height):
        for column_index in range(image_width):
            gx = gradient_x[row_index, column_index]
            gy = gradient_y[row_index, column_index]
            magnitude[i, j] = np.sqrt(gx * gx + gy * gy)

    return clip_float_to_uint8(magnitude)

### High-pass — Laplacian

Discrete 3×3 kernel; uses absolute filter response and rescales to `uint8`.

In [ ]:
# ==========================================================
# FILTRO LAPLACIAN (PASSA-ALTO)
# ==========================================================

def apply_laplacian_filter(input_image:
    """
    Laplacian discreto — single parameter: uint8 grayscale image.
    """

    if input_image.ndim != 2 or input_image.dtype != np.uint8:
        raise ValueError("Invalid input: expected 2D uint8 grayscale image.")

    kernel_laplacian = np.array(
        [
            [0.0, 1.0, 0.0],
            [1.0, -4.0, 1.0],
            [0.0, 1.0, 0.0],
        ],
        dtype=np.float64,
    )

    filter_response = apply_manual_convolution(input_image, kernel_laplacian)

    image_height, image_width = input_image.shape
    absolute_response = np.zeros((image_height, image_width), dtype=np.float64)

    for row_index in range(image_height):
        for column_index in range(image_width):
            absolute_response[i, j] = abs(filter_response[row_index, column_index])

    maximum_value = float(absolute_response.max())
    if maximum_value <= 0:
        return input_image.copy()

    scaled_response = (absolute_response / maximum_value) * 255.0
    return clip_float_to_uint8(scaled_response)